# DBLP Pipeline

## Parsing all Papers between 1985 and 2025

#### How does it works
##### Step 1:
We extract from DBLP (https://dblp.org/) all papers, including:
- title,
- authors, 
- venue
- doi (if accesible)
- abstract_link 
- venue
- pages
- type


In [1]:
import pandas as pd 
import numpy as np 
from helper.DBLP_Extractor import DBLPConferenceExtractor
import time
from datetime import datetime
from helper.clean import clean_html_entities, detect_german_titles
from typing import List, Dict, Optional

### Infos
- IJCAI know every year
- ECAI do not know each year
- AAAI same
- KI nearly every year (special case, that is why it is called independently)

In [2]:
conference_dict = {
   "IJCAI" : [1985, 1987, 1989, 1991, 1993, 1995, 1997, 1999, 2001, 2003, 2005, 2007, 2009, 2011, 2013, 2015, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
   "ECAI":[1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
   "AAAI": [1986, 1987, 1988, 1990, 1991, 1992, 1993, 1994, 1996, 1997, 1998, 1999, 2000, 2002, 2004, 2005, 2006, 2007, 2008, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
    "KI":[1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
    "GWAI":[1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992]
}

In [3]:
def get_papers_year(year: int, conference: str, extractor) -> pd.DataFrame:
    
    conference_config = {
        "IJCAI": {
            "query": "IJCAI",
            "venue_filter": ["IJCAI"],
            "venue_filter_mode": "exact"
        },
        "ECAI": {
            "query": "ECAI",
            "venue_filter": ["ECAI"],
            "venue_filter_mode": "exact"  
        },
        "AAAI": {
            "query": "AAAI",
            "venue_filter": ["AAAI"],
            "venue_filter_mode": "contains"
        },
        "KI": {
            "query": f"stream:conf/ki:",
            "venue_filter": ["KI"],
            "venue_filter_mode": "contains"
        },
        "GWAI": {
            "query": f"stream:conf/ki:",
            "venue_filter": ["GWAI"],
            "venue_filter_mode": "exact"
        }
    }
    
    config = conference_config.get(conference)
    if not config:
        print(f"⚠️  Unknown conference: {conference}")
        return None
    if conference not in ["GWAI", "KI"]:
        papers = extractor.fetch_conference_papers(
            conference_query=f"{config['query']} {year}",
            venue_filter=config['venue_filter'],
            venue_filter_mode=config['venue_filter_mode'],
            year_start=year,
            year_end=year,
            verbose=True,
            console_output=False  # Keine Console-Ausgabe, nur in Log
        )
    else:
        papers = extractor.fetch_conference_papers(
            conference_query=f"{config['query']}",
            venue_filter=config['venue_filter'],
            venue_filter_mode=config['venue_filter_mode'],
            year_start=year,
            year_end=year,
            verbose=True,
            console_output=False  # Keine Console-Ausgabe, nur in Log
        )
    
    if papers:
        df = pd.DataFrame(papers)
        df_clean = df[df['type'] != 'Editorship'].copy()
        return df_clean
    
    return None


In [24]:
def get_res(year_list: List[int], conference: str, log_file: str) -> Dict[int, pd.DataFrame]:
    # Erstelle Extractor mit Log-File
    extractor = DBLPConferenceExtractor(log_file=log_file)
    results = {}
    
    for year in year_list:
        extractor._log(f"\n>>> Processing {conference} {year}", console=True)  # Nur Jahr auf Console
        df = get_papers_year(year, conference, extractor)
        if df is not None and len(df) > 0:
            results[year] = df
        time.sleep(2)
    
    return results

In [25]:
ls_results = []

In [26]:
for conference, year_list in conference_dict.items():
    print(f"Verarbeite Konferenz: {conference}")
    log_file = f"dblp_extraction_{conference}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

    res = get_res(year_list, conference, log_file)
    ls_results.append(res)


    print(f"\n✓ {conference}: {len(res)}/{len(year_list)} Jahre gefunden")
    print(f"  Total Papers: {sum(len(df) for df in res.values())}")
    print(f"  Log gespeichert: {log_file}")


Verarbeite Konferenz: IJCAI

>>> Processing IJCAI 1985

>>> Processing IJCAI 1987

>>> Processing IJCAI 1989

>>> Processing IJCAI 1991

>>> Processing IJCAI 1993

>>> Processing IJCAI 1995

>>> Processing IJCAI 1997

>>> Processing IJCAI 1999

>>> Processing IJCAI 2001

>>> Processing IJCAI 2003

>>> Processing IJCAI 2005

>>> Processing IJCAI 2007

>>> Processing IJCAI 2009

>>> Processing IJCAI 2011

>>> Processing IJCAI 2013

>>> Processing IJCAI 2015

>>> Processing IJCAI 2017

>>> Processing IJCAI 2018

>>> Processing IJCAI 2019

>>> Processing IJCAI 2020

>>> Processing IJCAI 2021

>>> Processing IJCAI 2022

>>> Processing IJCAI 2023

>>> Processing IJCAI 2024

>>> Processing IJCAI 2025

✓ IJCAI: 25/25 Jahre gefunden
  Total Papers: 13395
  Log gespeichert: dblp_extraction_IJCAI_20260129_214831.log
Verarbeite Konferenz: ECAI

>>> Processing ECAI 1985

>>> Processing ECAI 1986

>>> Processing ECAI 1987

>>> Processing ECAI 1988

>>> Processing ECAI 1989

>>> Processing ECAI 1990


In [27]:
all_papers = []

for conference, year_list in conference_dict.items():
    conference_idx = list(conference_dict.keys()).index(conference)
    results_dict = ls_results[conference_idx]
    
    for year, df in results_dict.items():
        df_copy = df.copy()
        df_copy['conference'] = conference  # Füge Konferenz-Spalte hinzu
        all_papers.append(df_copy)

In [28]:
if all_papers:
    combined_df = pd.concat(all_papers, ignore_index=True)
    combined_df.to_csv("all_conferences_raw.csv", index=False)

## Clean & Evaluate

In [2]:
df_paper = pd.read_csv("all_conferences_raw.csv")

In [3]:
df_paper.columns

Index(['title', 'year', 'authors', 'doi', 'url', 'ee', 'venue', 'pages',
       'type', 'conference'],
      dtype='object')

In [4]:
df_paper['conference'] = df_paper['conference'].replace("GWAI", "KI")
df_paper['venue'] = df_paper['venue'].replace("GWAI", "KI")

In [5]:
def filter_conferences_by_paper_count(df, min_papers=10):
    allowed_venues = ['KI', 'ECAI', 'IJCAI', 'AAAI']
    filtered_df = df.groupby('conference').filter(lambda x: len(x) >= min_papers) 
    filtered_df_withdrawn = filtered_df[filtered_df['type'] != 'Withdrawn Items']
    filtered_df_venues = filtered_df_withdrawn[filtered_df_withdrawn['venue'].isin(allowed_venues)] 
    return filtered_df_venues

In [6]:
df_clean = filter_conferences_by_paper_count(df_paper, min_papers=50)
print(len(df_clean)/len(df_paper))

0.8759465922678358


In [7]:
df_html_clean = clean_html_entities(df_clean)
print(len(df_html_clean)/len(df_clean))

1.0


In [12]:
df_languages = detect_german_titles(df_html_clean, "KI")

Found German titles: 152


## Evaluate

In [11]:
def analyze_pages_by_conference_and_year(df):    
    def parse_pages(page_string):
        if pd.isna(page_string) or page_string == '':
            return np.nan
        
        try:
            page_string = str(page_string).strip()
            if '-' in page_string:
                parts = page_string.split('-')
                start = int(parts[0].strip())
                end = int(parts[1].strip())
                return end - start + 1
            else:
                return 1
        except:
            return np.nan
    
    df_copy = df.copy()
    df_copy['page_count'] = df_copy['pages'].apply(parse_pages)
    
    conference_year_stats = df_copy.groupby(['conference', 'year']).agg({
        'page_count': ['count', 'mean', 'median', 'std', 'min', 'max']
    }).round(2)
    
    conference_year_stats.columns = ['total_papers', 'mean_pages', 'median_pages', 'std_pages', 'min_pages', 'max_pages']
    conference_year_stats = conference_year_stats.reset_index()
    
    def flag_outliers(row):
        conf = row['conference']
        year = row['year']
        page_count = row['page_count']
        
        if pd.isna(page_count):
            return 'missing'
        
        stats = conference_year_stats[(conference_year_stats['conference'] == conf) & 
                                       (conference_year_stats['year'] == year)]
        
        if len(stats) == 0:
            return 'normal'
        
        conf_mean = stats['mean_pages'].values[0]
        conf_std = stats['std_pages'].values[0]
        
        if pd.notna(conf_std) and conf_std > 0:
            z_score = abs(page_count - conf_mean) / conf_std
            if z_score > 2:
                return 'outlier'
        
        return 'normal'
    
    df_copy['page_status'] = df_copy.apply(flag_outliers, axis=1)
    
    return conference_year_stats, df_copy



In [22]:
df = df_languages[['title','year','authors','doi', 'url','ee','venue','pages','conference', 'detected_lang']]

In [23]:
conference_year_stats, df_with_flags = analyze_pages_by_conference_and_year(df)
summary = df_with_flags.groupby(['conference', 'year', 'page_status']).size().unstack(fill_value=0)
print(summary)

print(f"Missing: {len(df_with_flags[df_with_flags['page_status'] == 'missing'])}")
print(f"Outliers: {len(df_with_flags[df_with_flags['page_status'] == 'outlier'])}")
print(f"Normal: {len(df_with_flags[df_with_flags['page_status'] == 'normal'])}")

page_status      missing  normal  outlier
conference year                          
AAAI       1986        2     181       16
           1987        0     145        4
           1988        1     145        4
           1990        0     165        9
           1991        0     133       11
...                  ...     ...      ...
KI         2021        0      25        2
           2022       14      16        1
           2023        0      19        0
           2024        0      31        0
           2025        0      27        1

[119 rows x 3 columns]
Missing: 336
Outliers: 2800
Normal: 40819


In [24]:
def find_missing_papers_by_page_gaps(df):
    def parse_page_range(page_string):
        if pd.isna(page_string) or page_string == '':
            return None, None
        
        try:
            page_string = str(page_string).strip()
            if '-' in page_string:
                parts = page_string.split('-')
                start = int(parts[0].strip())
                end = int(parts[1].strip())
                return start, end
            else:
                # Einzelne Seite
                page = int(page_string)
                return page, page
        except:
            return None, None
    
    df_copy = df.copy()
    
    df_copy[['page_start', 'page_end']] = df_copy['pages'].apply(
        lambda x: pd.Series(parse_page_range(x))
    )
    
    df_copy = df_copy.dropna(subset=['page_start', 'page_end'])
    df_copy['page_start'] = df_copy['page_start'].astype(int)
    df_copy['page_end'] = df_copy['page_end'].astype(int)
    df_copy['page_count'] = df_copy['page_end'] - df_copy['page_start'] + 1
    
    median_pages = df_copy.groupby(['conference', 'year'])['page_count'].median().reset_index()
    median_pages.columns = ['conference', 'year', 'median_page_count']
    
    gaps = []
    
    for (conf, year), group in df_copy.groupby(['conference', 'year']):
        group_sorted = group.sort_values('page_start').reset_index(drop=True)
        
        median = median_pages[
            (median_pages['conference'] == conf) & 
            (median_pages['year'] == year)
        ]['median_page_count'].values[0]
        
        for i in range(len(group_sorted) - 1):
            current_end = group_sorted.loc[i, 'page_end']
            next_start = group_sorted.loc[i + 1, 'page_start']
            
            gap_size = next_start - current_end - 1
            
            if gap_size >= median:
                potential_missing = int(gap_size / median)
                
                gaps.append({
                    'conference': conf,
                    'year': year,
                    'after_paper': group_sorted.loc[i, 'title'],
                    'before_paper': group_sorted.loc[i + 1, 'title'],
                    'gap_start': current_end + 1,
                    'gap_end': next_start - 1,
                    'gap_size': gap_size,
                    'median_pages': median,
                    'potential_missing_papers': potential_missing
                })
    
    gaps_df = pd.DataFrame(gaps)
    
    if len(gaps_df) > 0:
        gaps_df = gaps_df.sort_values(['conference', 'year', 'gap_start'])
    
    return gaps_df, median_pages

gaps_df, median_pages = find_missing_papers_by_page_gaps(df)
print(f"Found Gaps: {len(gaps_df)}")
print(gaps_df[['conference', 'year', 'gap_start', 'gap_end', 'gap_size', 'potential_missing_papers']])

Found Gaps: 61
   conference  year  gap_start  gap_end  gap_size  potential_missing_papers
0        AAAI  1986        836      842         7                         1
1        AAAI  1986        872      877         6                         1
2        AAAI  1986       1138     1145         8                         1
3        AAAI  2000       1065     1097        33                        33
4        AAAI  2000       1099     1104         6                         6
..        ...   ...        ...      ...       ...                       ...
56      IJCAI  2011       1085     2084      1000                       166
57      IJCAI  2013       2042     2048         7                         1
58      IJCAI  2015        974      988        15                         2
59      IJCAI  2015       2431     2444        14                         2
60      IJCAI  2020       4569     4575         7                         1

[61 rows x 6 columns]


In [25]:
print(gaps_df[['conference', 'year', 'gap_size', 'median_pages', 'potential_missing_papers']].sort_values('gap_size', ascending=False).head(10))


   conference  year  gap_size  median_pages  potential_missing_papers
56      IJCAI  2011      1000           6.0                       166
6        AAAI  2010       201           6.0                        33
11       AAAI  2012       181           7.0                        25
9        AAAI  2011       169           6.0                        28
12       AAAI  2013       155           7.0                        22
46      IJCAI  2005       100           6.0                        16
48      IJCAI  2007       100           6.0                        16
18       ECAI  1998       100           5.0                        20
39      IJCAI  1999       100           6.0                        16
17       AAAI  2020        58           8.0                         7


## Save Dataframe

In [26]:
df.head()

,title,year,authors,doi,url,ee,venue,pages,conference,detected_lang
0,A Representation for Complex Physical Domains.,1985,Sanjaya Addanki; Ernest Davis,NaN,https://dblp.org/rec/conf/ijcai/AddankiD85,http://ijcai.org/Proceedings/85-1/Papers/086.pdf,IJCAI,443-446,IJCAI,NaN
1,Shape from Texture.,1985,John Aliomonos; Michael J. Swain,NaN,https://dblp.org/rec/conf/ijcai/AliomonosS85,http://ijcai.org/Proceedings/85-2/Papers/051.pdf,IJCAI,926-931,IJCAI,NaN
2,Object Recognition Using Vision and Touch.,1985,Peter K. Allen; Ruzena Bajcsy,NaN,https://dblp.org/rec/conf/ijcai/AllenB85,http://ijcai.org/Proceedings/85-2/Papers/095.pdf,IJCAI,1131-1137,IJCAI,NaN
3,A Common-Sense Theory of Time.,1985,James F. Allen; Patrick J. Hayes,NaN,https://dblp.org/rec/conf/ijcai/AllenH85,http://ijcai.org/Proceedings/85-1/Papers/101.pdf,IJCAI,528-531,IJCAI,NaN
4,The Geometry Tutor.,1985,John R. Anderson; C. Franklin Boyle; Gregg Yost,NaN,https://dblp.org/rec/conf/ijcai/AndersonBY85,http://ijcai.org/Proceedings/85-1/Papers/001.pdf,IJCAI,1-7,IJCAI,NaN


In [27]:
df.to_csv("all_conferences_papers.csv", index=False)